# Sesión 2 — Ingesta de datos y capa Bronze  
## Notebook del estudiante actualizado con archivos reales

**Curso:** Databricks práctico — Lumi Commerce Lakehouse + Caso espejo Lluvia/Caña/Bagazo

### Objetivo
Cargar los archivos reales de Lumi y Bagazo en Databricks, crear tablas Bronze y construir una auditoría inicial.

### Reglas de la práctica
- Completa las celdas marcadas como `TODO`.
- No limpies todavía reglas de negocio.
- Valida columnas y conteos.
- Guarda tus cambios en GitHub al final.

## 0. Inventario real

Debes cargar los siguientes archivos:

### Lumi
- `olist_customers_dataset.csv` → tabla `customers` (99,441 filas esperadas)
- `olist_orders_dataset.csv` → tabla `orders` (99,441 filas esperadas)
- `olist_order_items_dataset.csv` → tabla `order_items` (112,650 filas esperadas)
- `olist_order_payments_dataset.csv` → tabla `order_payments` (103,886 filas esperadas)
- `olist_order_reviews_dataset.csv` → tabla `order_reviews` (99,224 filas esperadas)
- `olist_products_dataset.csv` → tabla `products` (32,951 filas esperadas)
- `olist_sellers_dataset.csv` → tabla `sellers` (3,095 filas esperadas)
- `olist_geolocation_dataset.csv` → tabla `geolocation` (1,000,163 filas esperadas)
- `product_category_name_translation.csv` → tabla `product_category_translation` (71 filas esperadas)

### Bagazo
- `molienda_bagazo_y_lluvias_II.csv` → tabla `molienda_bagazo_y_lluvias` (798 filas esperadas)

In [0]:
# =========================================
# 1. Validación del entorno
# =========================================
from datetime import datetime
from typing import Optional
import re
from pyspark.sql import functions as F

print("Spark version:", spark.version)
current_catalog = spark.sql("SELECT current_catalog() AS catalog").collect()[0]["catalog"]
print("Current catalog:", current_catalog)

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.raw")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.raw.lumi")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.raw.bagazo")

In [0]:
# =========================================
# 2. Parámetros de trabajo
# =========================================
CATALOG = current_catalog

LUMI_BRONZE_SCHEMA = "lumi_bronze"
BAGAZO_BRONZE_SCHEMA = "bagazo_bronze"
CONTROL_SCHEMA = "control"

# TODO 1:
# Cambia estas rutas según dónde subiste los archivos.
LUMI_RAW_PATH = "/Volumes/workspace/raw/lumi"
BAGAZO_RAW_PATH = "/Volumes/workspace/raw/bagazo"

print("LUMI_RAW_PATH:", LUMI_RAW_PATH)
print("BAGAZO_RAW_PATH:", BAGAZO_RAW_PATH)

display(dbutils.fs.ls(LUMI_RAW_PATH))
display(dbutils.fs.ls(BAGAZO_RAW_PATH))

LUMI_FILES = {
    "customers": "olist_customers_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "product_category_translation": "product_category_name_translation.csv"
}

BAGAZO_FILE = "molienda_bagazo_y_lluvias_II.csv"
BAGAZO_DATASET = "molienda_bagazo_y_lluvias_II"

EXPECTED_ROWS = globals().get("EXPECTED_ROWS", {})
EXPECTED_COLUMNS = globals().get("EXPECTED_COLUMNS", {})

# Valores esperados para Lumi
EXPECTED_ROWS["customers"] = 99441
EXPECTED_ROWS["orders"] = 99441
EXPECTED_ROWS["order_items"] = 112650
EXPECTED_ROWS["order_payments"] = 103886
EXPECTED_ROWS["order_reviews"] = 99224
EXPECTED_ROWS["products"] = 32951
EXPECTED_ROWS["sellers"] = 3095
EXPECTED_ROWS["geolocation"] = 1000163
EXPECTED_ROWS["product_category_translation"] = 71

# Valores esperados para Bagazo
EXPECTED_ROWS[BAGAZO_DATASET] = 798

EXPECTED_COLUMNS[BAGAZO_DATASET] = [
    "Fecha",
    "PROMEDIO LLUVIAS  PROVIDENCIA (mm)",
    "CAÑA MOLIDA PROVIDENCIA (Toneladas)",
    "BAGAZO ENTREGADO PROVIDENCIA (Toneladas)",
    "Comentarios PROVIDENCIA",
    "PROMEDIO LLUVIAS ILC (mm)",
    "CAÑA MOLIDA ILC (Toneladas)",
    "BAGAZO ENTREGADO ILC (Toneladas)",
    "comentarios ILC",
    "PROMEDIO LLUVIAS INCAUCA  (mm)",
    "CAÑA MOLIDA INCAUCA (Toneladas)",
    "BAGAZO ENTREGADO INCAUCA (Toneladas)",
    "comentarios INCAUCA"
]

EXPECTED_COLUMNS = {
    "customers": [
        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state"
    ],
    "orders": [
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ],
    "order_items": [
        "order_id",
        "order_item_id",
        "product_id",
        "seller_id",
        "shipping_limit_date",
        "price",
        "freight_value"
    ],
    "order_payments": [
        "order_id",
        "payment_sequential",
        "payment_type",
        "payment_installments",
        "payment_value"
    ],
    "order_reviews": [
        "review_id",
        "order_id",
        "review_score",
        "review_comment_title",
        "review_comment_message",
        "review_creation_date",
        "review_answer_timestamp"
    ],
    "products": [
        "product_id",
        "product_category_name",
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ],
    "sellers": [
        "seller_id",
        "seller_zip_code_prefix",
        "seller_city",
        "seller_state"
    ],
    "geolocation": [
        "geolocation_zip_code_prefix",
        "geolocation_lat",
        "geolocation_lng",
        "geolocation_city",
        "geolocation_state"
    ],
    "product_category_translation": [
        "product_category_name",
        "product_category_name_english"
    ],
    "molienda_bagazo_y_lluvias": [
        "Fecha",
        "PROMEDIO LLUVIAS  PROVIDENCIA (mm)",
        "CAÑA MOLIDA PROVIDENCIA (Toneladas)",
        "BAGAZO ENTREGADO PROVIDENCIA (Toneladas)",
        "Comentarios\nPROVIDENCIA",
        "PROMEDIO LLUVIAS ILC (mm)",
        "CAÑA MOLIDA ILC (Toneladas)",
        "BAGAZO ENTREGADO ILC (Toneladas)",
        "Comentarios\nILC",
        "PROMEDIO LLUVIAS INCAUCA  (mm)",
        "CAÑA MOLIDA INCAUCA (Toneladas)",
        "BAGAZO ENTREGADO INCAUCA (Toneladas)",
        "Comentarios\nINCAUCA"
    ]
}

print("Revisa tus rutas:")
print("LUMI_RAW_PATH:", LUMI_RAW_PATH)
print("BAGAZO_RAW_PATH:", BAGAZO_RAW_PATH)

## 1. Crear esquemas

Crea los esquemas:

- `lumi_bronze`
- `bagazo_bronze`
- `control`

In [0]:
# TODO 2:
# Crea los esquemas usando spark.sql("CREATE SCHEMA IF NOT EXISTS ...")

for schema in [LUMI_BRONZE_SCHEMA, BAGAZO_BRONZE_SCHEMA, CONTROL_SCHEMA]:
    # TODO: completa la instrucción SQL
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{schema}`")
    pass

## 2. Funciones auxiliares

Completa las funciones básicas de lectura, normalización y auditoría.

In [0]:
# =========================================
# 4. Funciones auxiliares
# =========================================
def normalize_column_name(name: str) -> str:
    replacements = {
        "á": "a", "é": "e", "í": "i", "ó": "o", "ú": "u", "ñ": "n",
        "Á": "a", "É": "e", "Í": "i", "Ó": "o", "Ú": "u", "Ñ": "n"
    }
    for old, new in replacements.items():
        name = name.replace(old, new)
    name = name.replace("\n", "_").replace("\r", "_")
    name = name.strip().lower()
    name = re.sub(r"[^a-zA-Z0-9_]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name


def normalize_columns(df):
    new_cols = [normalize_column_name(c) for c in df.columns]
    return df.toDF(*new_cols)


def read_csv_safely(path: str, sep: str = ",", infer_schema: bool = True, normalize_cols: bool = True):
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", infer_schema)
        .option("sep", sep)
        .option("multiLine", True)
        .option("quote", '"')
        .option("escape", '"')
        .csv(path)
    )
    if normalize_cols:
        df = normalize_columns(df)
    return df


def normalized_expected_columns(dataset_name: str):
    return [normalize_column_name(c) for c in EXPECTED_COLUMNS[dataset_name]]


def validate_columns(df, dataset_name: str):
    expected = set(normalized_expected_columns(dataset_name))
    actual = set(df.columns)
    missing = sorted(list(expected - actual))
    extra = sorted(list(actual - expected))
    if missing:
        print(f"⚠️ Columnas faltantes en {dataset_name}: {missing}")
    if extra:
        print(f"ℹ️ Columnas adicionales en {dataset_name}: {extra}")
    if not missing:
        print(f"✅ Columnas esperadas presentes en {dataset_name}")
    return missing, extra


def write_delta_table(df, table_fqn: str, mode: str = "overwrite"):
    (
        df.write
        .format("delta")
        .mode(mode)
        .option("overwriteSchema", "true")
        .saveAsTable(table_fqn)
    )
    print(f"Tabla escrita: {table_fqn}")


audit_rows = []


def register_audit(dataset: str, archivo_origen: str, tabla_destino: str, filas_cargadas: Optional[int], columnas: Optional[int], estado: str, observaciones: str):
    audit_rows.append({
        "dataset": dataset,
        "archivo_origen": archivo_origen,
        "tabla_destino": tabla_destino,
        "filas_cargadas": int(filas_cargadas) if filas_cargadas is not None else None,
        "columnas": int(columnas) if columnas is not None else None,
        "fecha_carga": datetime.now().isoformat(timespec="seconds"),
        "estado": estado,
        "observaciones": observaciones
    })

In [0]:
audit_rows = []

# TODO 4:
# Completa función de auditoría.
def register_audit(dataset, archivo_origen, tabla_destino, filas_cargadas, columnas, estado, observaciones):
    audit_rows.append({
        "dataset": dataset,
        "archivo_origen": archivo_origen,
        "tabla_destino": tabla_destino,
        "filas_cargadas": filas_cargadas,
        "columnas": columnas,
        "fecha_carga": datetime.now().isoformat(timespec="seconds"),
        "estado": estado,
        "observaciones": observaciones
    })

## 3. Reto guiado: carga de un archivo Lumi

Carga primero `customers`.

### Resultado esperado
- Tabla: `lumi_bronze.customers`
- Filas esperadas: 99,441
- Columnas esperadas: 5

In [0]:
# TODO 5:
# Carga customers, cuenta filas y columnas, guarda como tabla Delta.

dataset = "customers"
filename = LUMI_FILES[dataset]
path = f"{LUMI_RAW_PATH}/{filename}"
table_fqn = f"`{CATALOG}`.`{LUMI_BRONZE_SCHEMA}`.`{dataset}`"

df_customers = read_csv_safely(path)
filas = df_customers.count()
columnas = len(df_customers.columns)
df_customers.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_fqn)

# display(df_customers.limit(10))
# print(filas, columnas)

## 4. Carga de todos los archivos Lumi

Ahora automatiza la carga de los 9 datasets.

In [0]:
# TODO 6:
# Recorre LUMI_FILES y carga cada archivo en lumi_bronze.

for dataset, filename in LUMI_FILES.items():
    path = f"{LUMI_RAW_PATH}/{filename}"
    table_fqn = f"`{CATALOG}`.`{LUMI_BRONZE_SCHEMA}`.`{dataset}`"

    # TODO:
    # 1. leer archivo
    # 2. contar filas
    # 3. contar columnas
    # 4. escribir tabla Delta
    # 5. registrar auditoría
    pass

In [0]:
loaded_lumi = {}
 
for dataset, filename in LUMI_FILES.items():
    path = f"{LUMI_RAW_PATH}/{filename}"
    table_fqn = f"`{CATALOG}`.`{LUMI_BRONZE_SCHEMA}`.`{dataset}`"
 
    print("=" * 100)
    print(f"Cargando dataset: {dataset}")
    print(f"Archivo: {path}")
    print(f"Destino: {table_fqn}")
 
    try:
        df = read_csv_safely(path)
        missing, extra = validate_columns(df, dataset)
        rows = df.count()
        cols = len(df.columns)
 
        expected_rows = EXPECTED_ROWS[dataset]
        if rows == expected_rows:
            row_msg = f"Conteo OK: {rows:,} filas"
            print(f"✅ {row_msg}")
        else:
            row_msg = f"Conteo diferente. Esperado: {expected_rows:,}; obtenido: {rows:,}"
            print(f"⚠️ {row_msg}")
 
        write_delta_table(df, table_fqn)
        loaded_lumi[dataset] = df
 
        estado = "OK" if not missing and rows == expected_rows else "REVISAR"
        observaciones = f"{row_msg}. Columnas: {cols}. Faltantes: {missing}. Extras: {extra}"
        register_audit(dataset, filename, f"{CATALOG}.{LUMI_BRONZE_SCHEMA}.{dataset}", rows, cols, estado, observaciones)
 
    except Exception as e:
        print(f"❌ Error cargando {dataset}: {e}")
        register_audit(dataset, filename, f"{CATALOG}.{LUMI_BRONZE_SCHEMA}.{dataset}", None, None, "ERROR", str(e))

## 5. Validación de conteos Lumi

Valida que cada tabla tenga el conteo esperado.

In [0]:
# TODO 7:
# Construye un resumen con:
# dataset, filas_obtenidas, filas_esperadas, columnas, estado

summary_rows = []

for dataset in LUMI_FILES.keys():
    table_fqn = f"`{CATALOG}`.`{LUMI_BRONZE_SCHEMA}`.`{dataset}`"
    # TODO:
    # df = spark.table(table_fqn)
    # rows = df.count()
    # cols = len(df.columns)
    # expected = EXPECTED_ROWS[dataset]
    # estado = "OK" if rows == expected else "REVISAR"
    # summary_rows.append((dataset, rows, expected, cols, estado))
    pass

# summary_df = spark.createDataFrame(summary_rows, ["dataset", "filas_obtenidas", "filas_esperadas", "columnas", "estado"])
# display(summary_df)

## 6. Carga de Bagazo

Carga `molienda_bagazo_y_lluvias_II.csv`.

### Resultado esperado
- Tabla: `bagazo_bronze.molienda_bagazo_y_lluvias`
- Filas esperadas: 798
- Columnas esperadas: 13
- Rango de fechas esperado: 2024-01-01 a 2026-03-08

In [0]:
# TODO 8:
# Carga el dataset Bagazo.

bagazo_dataset = "molienda_bagazo_y_lluvias"
bagazo_path = f"{BAGAZO_RAW_PATH}/{BAGAZO_FILE}"
bagazo_table_fqn = f"`{CATALOG}`.`{BAGAZO_BRONZE_SCHEMA}`.`{bagazo_dataset}`"

# bagazo_df = read_csv_safely(bagazo_path)
# filas = bagazo_df.count()
# columnas = len(bagazo_df.columns)
# bagazo_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(bagazo_table_fqn)
# display(bagazo_df.limit(10))

## 7. Crear tabla de auditoría

Materializa `control.audit_ingestion_sesion_02`.

In [0]:
# TODO 9:
# Crea un DataFrame de auditoría y guárdalo como tabla Delta.

# audit_df = spark.createDataFrame(audit_rows)
# audit_table_fqn = f"`{CATALOG}`.`{CONTROL_SCHEMA}`.`audit_ingestion_sesion_02`"
# audit_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(audit_table_fqn)
# display(spark.table(audit_table_fqn))

## 8. Retos

### Reto Nivel 1 — Validación básica
Confirma que todas las tablas existen y tienen el conteo esperado.

### Reto Nivel 2 — Auditoría completa
Agrega a la auditoría una columna adicional llamada `tipo_fuente` con valores `lumi_csv` o `bagazo_csv`.

### Reto consultor — Riesgos de producción
Responde:

1. ¿Qué cambiarías si esto fuera Azure Databricks empresarial?
2. ¿Dónde pondrías los archivos en vez de DBFS/FileStore?
3. ¿Qué validaciones agregarías antes de pasar a Silver?
4. ¿Qué riesgos ves en el dataset Bagazo por estar en formato ancho?

## 9. Conclusión personal

Completa:

- Lo más importante que aprendí hoy fue:
- El error que debo evitar en una ingesta Bronze es:
- La evidencia que mostraría para probar que una carga fue exitosa es:
- En una empresa real, esta ingesta debería automatizarse con: